# 12.12 - Embeddings & Vector Stores

**Phase:** 12 - LangChain

**Status:** VERIFIED

---

## 1. What Are We Solving?

To retrieve relevant documents for RAG, we need to convert text to vectors.

## 2. Why Does This Matter?

Embeddings capture semantic meaning. Vector stores enable fast retrieval.

## 3. Prerequisites

- 12.10: Document loaders
- 12.11: Text splitters

## 4. Learning Objectives

- Understand text embeddings
- Build a simple vector store
- Perform similarity search
- Understand TF-IDF and cosine similarity

## 5. Mental Model

Embedding = text -> vector.
Vector store = database of vectors + metadata.
Similarity search = find vectors closest to query.

In [1]:
import numpy as np
from langchain_core.documents import Document
print("Libraries loaded.")

Libraries loaded.


## 6. What Are Embeddings?

In [2]:
print("Embeddings convert text to vectors (lists of numbers).")
print("Similar texts have similar vectors.")
print("")
print("Example:")
print("  \"The cat sat\" -> [0.2, 0.8, 0.1, ...]")
print("  \"A feline rested\" -> [0.3, 0.7, 0.2, ...] (similar!)")
print("  \"The car drove\" -> [0.9, 0.1, 0.6, ...] (different)")

Embeddings convert text to vectors (lists of numbers).
Similar texts have similar vectors.

Example:
  "The cat sat" -> [0.2, 0.8, 0.1, ...]
  "A feline rested" -> [0.3, 0.7, 0.2, ...] (similar!)
  "The car drove" -> [0.9, 0.1, 0.6, ...] (different)


## 7. TF-IDF Embeddings

In [3]:
from collections import Counter
import math

def tfidf_embed(texts):
    """Compute TF-IDF vectors for a list of texts."""
    # Tokenize
    tokenized = [text.lower().split() for text in texts]
    
    # Build vocabulary
    vocab = set()
    for tokens in tokenized:
        vocab.update(tokens)
    vocab = sorted(vocab)
    word_to_idx = {w: i for i, w in enumerate(vocab)}
    
    # Compute IDF
    n = len(texts)
    df = Counter()
    for tokens in tokenized:
        for w in set(tokens):
            df[w] += 1
    
    idf = {}
    for word in vocab:
        idf[word] = math.log((n + 1) / (df.get(word, 0) + 1)) + 1
    
    # Compute TF-IDF vectors
    vectors = []
    for tokens in tokenized:
        tf = Counter(tokens)
        total = len(tokens)
        vec = np.zeros(len(vocab))
        for word, count in tf.items():
            if word in word_to_idx:
                vec[word_to_idx[word]] = (count / total) * idf[word]
        vectors.append(vec)
    
    return np.array(vectors), vocab

# Test
texts = ["the cat sat on the mat", "the dog played in the park", "the cat chased the dog"]
vectors, vocab = tfidf_embed(texts)
print("Vocab size:", len(vocab))
print("Vector shape:", vectors.shape)

Vocab size: 10
Vector shape: (3, 10)


## 8. Cosine Similarity

In [4]:
def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    dot = np.dot(a, b)
    norm = np.linalg.norm(a) * np.linalg.norm(b)
    if norm == 0:
        return 0.0
    return float(dot / norm)

# Compare
print("sim(cat sat, dog played):", round(cosine_similarity(vectors[0], vectors[1]), 4))
print("sim(cat sat, cat chased):", round(cosine_similarity(vectors[0], vectors[2]), 4))
print("sim(dog played, cat chased):", round(cosine_similarity(vectors[1], vectors[2]), 4))

sim(cat sat, dog played):

 0.2805
sim(cat sat, cat chased): 0.4696
sim(dog played, cat chased): 0.4696


## 9. Simple Vector Store

In [5]:
class SimpleVectorStore:
    def __init__(self):
        self.docs = []
        self.vectors = None
        self.vocab = None
    
    def add_documents(self, docs):
        self.docs.extend(docs)
        texts = [doc.page_content for doc in self.docs]
        self.vectors, self.vocab = tfidf_embed(texts)
    
    def similarity_search(self, query, k=2):
        # Embed query using same vocab
        tokens = query.lower().split()
        from collections import Counter
        import math
        tf = Counter(tokens)
        total = len(tokens)
        n = len(self.docs)
        
        # Compute IDF for query words
        df = Counter()
        texts = [doc.page_content for doc in self.docs]
        for text in texts:
            for w in set(text.lower().split()):
                df[w] += 1
        
        idf = {}
        for word in self.vocab:
            idf[word] = math.log((n + 1) / (df.get(word, 0) + 1)) + 1
        
        # Build query vector
        word_to_idx = {w: i for i, w in enumerate(self.vocab)}
        query_vec = np.zeros(len(self.vocab))
        for word, count in tf.items():
            if word in word_to_idx:
                query_vec[word_to_idx[word]] = (count / total) * idf.get(word, 1)
        
        # Find similar
        scores = []
        for i, vec in enumerate(self.vectors):
            score = cosine_similarity(query_vec, vec)
            scores.append((self.docs[i], score))
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:k]

store = SimpleVectorStore()

docs = [
    Document(page_content="LangChain is a framework for LLM applications.", metadata={"source": "doc1"}),
    Document(page_content="ChromaDB is a vector database for embeddings.", metadata={"source": "doc2"}),
    Document(page_content="RAG combines retrieval with generation.", metadata={"source": "doc3"}),
    Document(page_content="Python is the most popular language for ML.", metadata={"source": "doc4"}),
    Document(page_content="FastAPI is great for building APIs.", metadata={"source": "doc5"})
]

store.add_documents(docs)
print("Stored " + str(len(store.docs)) + " documents")

Stored 5 documents


## 10. Similarity Search

In [6]:
results = store.similarity_search("What is LangChain?", k=2)
print("Query: What is LangChain?")
for doc, score in results:
    print("  Score: " + str(round(score, 4)) + " - " + doc.page_content)
    print("    Source:", doc.metadata)

Query: What is LangChain?
  Score: 0.2617 - FastAPI is great for building APIs.
    Source: {'source': 'doc5'}
  Score: 0.245 - LangChain is a framework for LLM applications.
    Source: {'source': 'doc1'}


In [7]:
results = store.similarity_search("vector database", k=2)
print("Query: vector database")
for doc, score in results:
    print("  Score: " + str(round(score, 4)) + " - " + doc.page_content)

Query: vector database
  Score: 0.6151 - ChromaDB is a vector database for embeddings.
  Score: 0.0 - LangChain is a framework for LLM applications.


## 11. Common Mistakes

1. Wrong embedding model for your data
2. Chunks too large for embedding
3. Not persisting the vector store
4. Too few documents for meaningful search

## 12. Coding Exercises

### Exercise 1: Build a Vector Store
Create a vector store from text files.

### Exercise 2: Search Quality
Test different queries and evaluate results.

In [8]:
# EXERCISE 1
print("Exercise: Build a vector store from files.")

Exercise: Build a vector store from files.


In [9]:
# EXERCISE 2
print("Exercise: Test search quality with different queries.")

Exercise: Test search quality with different queries.


## 13. Closed-Book Recall

1. What is an embedding?
2. What does a vector store store?
3. How does similarity search work?

## 14. Summary

Embeddings convert text to vectors. Vector stores store and retrieve vectors. Similarity search finds relevant documents using cosine similarity.

## Verification Status
```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: [numpy, langchain-core]
OUTPUTS: PASS
LAST VERIFIED: 2026-08-30
```